# E2 · Parameter learning with missing data

Compare complete-case MLE, mode imputation followed by MLE, and EM on DAG. All methods use the same encoded splits, state order, and add-one smoothing. Each rate and seed generates one training mask shared by all three methods.

We add 0%, 20%, or 40% missingness independently among observed cells in the two condition-related columns. The 0% setting retains the original missing values. The three seeds repeat the mask draw on the same split, so their variability describes sensitivity to masking.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Find the repository root from wherever the notebook is launched.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dataset" / "prepared").is_dir())
sys.path.insert(0, str(root))

from missing_data.run_e2 import MASK_COLUMNS, RATES, SEEDS, prepare, run_comparison, summarize

In [ ]:
setup = prepare()
print(f"Training rows: {len(setup.train):,} | Test rows: {len(setup.test):,}")
display(setup.train[MASK_COLUMNS].isna().mean().rename("original_train_missing_fraction"))

## Fit and score one shared mask

Complete-case Maximum Likelihood Estimation (MLE) removes training rows containing any missing model variables. Mode imputation + MLE fills in missing training values using the most common observed states in each column, breaking ties based on state codes. The EM algorithm starts with the CPT of Mode imputation + MLE and directly fits the masked observations. We record its stopping state and number of iterations.

Test Log-Likelihood uses the same raw test evidence for each model and marginalizes missing values. To improve recovery accuracy, we create a separate fixed 20% test mask. Manually hidden columns remain masked when querying the posterior distribution. Recovery scores only compute intentionally hidden test cells with known raw values.

Parameter Diagnostics computes the average KL divergence between nodes and their parent configurations and the unmasked full-case reference CPT. The reference value is an estimate based on the observed data, and the raw missing values are inaccessible. A smaller KL value indicates higher agreement with the reference value.

## Compare results across masking seeds

Higher test LL and recovery accuracy are better. Lower CPT KL means closer agreement with the reference. At 0% extra masking, the three seeds use identical training evidence and the same EM initialization. For 20% and 40%, the paired differences count how often EM improves on complete-case MLE under the same mask.

In [ ]:
runs = []
diagnostics = []
for rate in RATES:
    for seed in SEEDS:
        scores, diagnostic = run_comparison(setup, rate, seed)
        runs.extend(scores)
        diagnostics.append(diagnostic)
        print(f"Rate={rate:.0%}, seed={seed}: {diagnostic['em_iterations']} EM iterations")

In [ ]:
results = pd.DataFrame(runs)
summary, paired_wins = summarize(results)
display(summary.set_index(["mask_rate", "method"]).round(5))
display(pd.DataFrame(diagnostics).round(4))

In [ ]:
display(paired_wins.set_index("mask_rate"))

## E2 conclusion

We ran E2 to measure how missing service history and condition scores affect parameter learning in our used-car Bayesian network. The learned conditional probability tables (CPTs) support downstream inference, including queries about listing-price tiers. We held the expert DAG, training masks, and test evidence fixed across methods so the comparison isolates how each estimator handles missing values.

The masking probabilities apply independently to observed cells in two training columns. At 0% additional masking, the original missing values remain. Test log-likelihood is averaged across three masking seeds and reported in nats per row. Higher is better.

| Added masking | Complete-case MLE | Mode imputation + MLE | EM | Complete-case training rows retained |
| --- | ---: | ---: | ---: | ---: |
| 0% | -10.86407 | -10.86682 | **-10.86374** | 76,081 / 80,000 (95.1%) |
| 20% | -10.86595 | -10.99005 | **-10.86431** | 48,699 / 80,000 (60.9%) |
| 40% | -10.87056 | -11.22263 | **-10.86666** | 27,441 / 80,000 (34.3%) |

- **0% added masking.** Complete-case MLE is a simple baseline and retains most rows. EM gains only 0.00033 nats per row over it and converges in three iterations. Mode imputation keeps all rows, with a slightly lower test likelihood.
- **20% added masking.** Complete-case MLE remains close to EM despite discarding nearly two in five rows. EM uses the observed values in partial rows and achieves higher test likelihood and lower CPT KL than complete-case MLE in all three seeds. Mode imputation retains all rows, although its test likelihood falls more sharply.
- **40% added masking.** Complete-case MLE still scores close to EM with only about a third of the training rows. EM again improves both test likelihood and CPT KL over complete-case MLE in all three seeds and takes 13–14 iterations to converge.


### Overall condition score recovery accuracy

`overall_condition_score_accuracy` is the fraction of deliberately hidden test entries whose original condition-score bin matches the model's most probable bin. A value of 0.74 means 74% of those entries were recovered correctly. All methods use the same separate 20% test mask. The rows below vary the additional masking during training, and each entry is the mean across three training-mask seeds.

| Added training masking | Complete-case MLE | Mode imputation + MLE | EM |
| --- | ---: | ---: | ---: |
| 0% | 74.06% | 74.06% | 73.96% |
| 20% | 74.02% | 73.53% | 74.04% |
| 40% | 74.02% | 59.84% | 74.13% |

Missingness still affects the fitted models. From 0% to 40%, test log-likelihood falls by 0.00649 nats per row for complete-case MLE, 0.35581 for mode imputation + MLE, and 0.00292 for EM. **EM shows the smallest decline among the three methods tested.** Its gain over complete-case MLE is modest for test likelihood and individual value recovery. CPT KL measures agreement with a reference estimate fitted before additional masking. The results describe random masking of these two columns under the expert DAG. Different columns or missingness mechanisms could change the comparison.